# 🏺 MummyBot — Chatbot

Ollama + FAISS retrieval. Run cells in order.

## Cell 1 — Imports & Config

In [1]:
import re, pickle
import numpy as np
import faiss
import ollama
from pathlib import Path
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown

INDEX_FILE  = Path("faiss_index.bin")
META_FILE   = Path("faiss_meta.pkl")
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
LLM_MODEL   = "llama3.2"

# Check Ollama
try:
    models = [m["name"] for m in ollama.list()["models"]]
    print(f"✅ Ollama running — models: {models}")
except:
    print("❌ Ollama not running — run `ollama serve`")

print(f"Index : {INDEX_FILE} — exists: {INDEX_FILE.exists()}")
print(f"Meta  : {META_FILE}  — exists: {META_FILE.exists()}")


❌ Ollama not running — run `ollama serve`
Index : faiss_index.bin — exists: True
Meta  : faiss_meta.pkl  — exists: True


## Cell 2 — Load FAISS + BGE

In [ ]:
print("Loading BGE model...")
model = SentenceTransformer(EMBED_MODEL)

print("Loading FAISS index...")
index = faiss.read_index(str(INDEX_FILE))

print("Loading metadata...")
with open(META_FILE, "rb") as f:
    chunks = pickle.load(f)

print(f"✅ {index.ntotal} vectors loaded")
print(f"✅ {len(chunks)} chunks loaded")


## Cell 3 — Query Rewriter + Retrieval

In [ ]:
QUERY_TOPIC_MAP = {
    "IDENTITY":  ["who is", "who was", "tell me about", "what was"],
    "MONUMENTS": ["built", "build", "construct", "pyramid", "temple", "monument", "achieve"],
    "DEATH":     ["die", "died", "death", "killed", "how did", "cause", "buried", "tomb"],
    "REIGN":     ["rule", "reign", "campaign", "battle", "war", "fight", "battles"],
    "FAMILY":    ["children", "wife", "husband", "son", "daughter", "family", "married"],
    "LEGACY":    ["legacy", "famous", "known for", "remembered", "impact"],
}

PHARAOH_NAMES = sorted([
    "ramesses ii", "ramesses iii", "ramesses iv", "ramesses ix", "ramesses xi", "ramesses i",
    "thutmose iii", "thutmose iv", "thutmose ii", "thutmose i",
    "amenhotep iii", "amenhotep ii", "amenhotep i",
    "ptolemy iii", "ptolemy ii", "ptolemy i",
    "psamtik iii", "psamtik ii", "psamtik i",
    "pepi ii", "pepi i", "seti ii", "seti i",
    "mentuhotep iv", "mentuhotep iii", "mentuhotep ii",
    "senusret iii", "senusret ii", "senusret i",
    "amenemhat iii", "amenemhat ii", "amenemhat i",
    "nectanebo ii", "nectanebo i", "shoshenq iii", "shoshenq i",
    "khufu", "khafre", "menkaure", "snefru", "djedefre", "shepseskaf",
    "hatshepsut", "akhenaten", "tutankhamun", "horemheb", "nefertiti",
    "ahmose", "merneptah", "cleopatra vii", "cleopatra", "arsinoe",
    "djoser", "piye", "taharqa", "shabaka", "shebitku", "tantamani",
    "narmer", "tawosret", "kamose", "seqenenre",
    "cambyses", "darius", "xerxes", "artaxerxes",
    "osorkon", "psusennes", "pinedjem", "smendes",
    "userkaf", "sahure", "niuserre", "unas", "teti",
    "sobekneferu", "huni", "sekhemkhet",
], key=len, reverse=True)

def detect_topic(query):
    q = query.lower()
    for topic, keywords in QUERY_TOPIC_MAP.items():
        if any(kw in q for kw in keywords):
            return topic
    return None

def extract_pharaoh(query):
    q = query.lower()
    for name in PHARAOH_NAMES:
        if name in q:
            roman = {"i","ii","iii","iv","v","vi","vii","viii","ix","x","xi"}
            return " ".join(p.upper() if p in roman else p.title() for p in name.split())
    return None

def retrieve(query, k=2, verbose=False):
    bge_q   = f"Represent this sentence for searching relevant passages: {query}"
    topic   = detect_topic(query)
    pharaoh = extract_pharaoh(query)

    if verbose:
        print(f"  Topic   : {topic}")
        print(f"  Pharaoh : {pharaoh}")

    q_vec = model.encode([bge_q], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_vec, k * 10)
    candidates = [(chunks[i], float(scores[0][j]))
                  for j, i in enumerate(indices[0]) if i >= 0]

    # Exact pharaoh match — dump anything that doesn't belong
    if pharaoh:
        matched = [(c, s) for c, s in candidates
                   if c["pharaoh"].lower() == pharaoh.lower()]
        if matched:
            candidates = matched

    # Topic filter
    if topic:
        topic_matched = [(c, s) for c, s in candidates
                         if c["topic"] == topic]
        if topic_matched:
            candidates = topic_matched

    candidates.sort(key=lambda x: x[1], reverse=True)

    if verbose:
        print(f"  Results : {len(candidates)} after filtering")

    return [c for c, s in candidates[:k]]

print("✅ Retrieval loaded")
print("\nTest:")
for q in ["What did Ramesses II build?", "How did Cleopatra die?", "Who was Khufu?"]:
    r = retrieve(q, k=1)
    pharaoh = r[0]["pharaoh"] if r else "none"
    topic   = r[0]["topic"]   if r else "none"
    print(f"  {q:40} → [{pharaoh}] [{topic}]")


## Cell 4 — System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are MummyBot — a friendly ancient Egyptian history guide for tourists.
Answer using ONLY the CONTEXT provided. Never use outside knowledge.

Format your answer as:
## [Pharaoh Name]
**Who:** one sentence
**What they did:** one or two sentences
**Key facts:**
- fact from context
- fact from context
**In summary:** one plain sentence

If context does not contain the answer:
"I don't have that in my Knowledge yet!"

If question is not about ancient Egypt:
"I only know about ancient Egyptian history!"
"""
print("✅ System prompt loaded")


## Cell 5 — Ask (Single Query + Stream)

Shows retrieved chunks then streams the answer.

In [ ]:
def ask(query, show_chunks=True):
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    docs    = retrieve(query, k=2, verbose=True)
    context = "\n\n---\n\n".join(d["text"] for d in docs)

    if show_chunks:
        print(f"\n── Chunks sent to LLM ───────────────────────")
        for i, d in enumerate(docs):
            print(f"  [{i+1}] [{d['pharaoh']:20}] [{d['topic']:10}] {d['text'][len(d['text'].split(']')[0])+1:][:80]}...")
        print()

    messages = [
        {"role": "system", "content": f"{SYSTEM_PROMPT}\n\nCONTEXT:\n{context}"},
        {"role": "user",   "content": query}
    ]

    print("── Answer ───────────────────────────────────")
    answer = ""
    for chunk in ollama.chat(model=LLM_MODEL, messages=messages, stream=True,
                              options={"num_predict": 400, "temperature": 0.2}):
        token = chunk["message"]["content"]
        print(token, end="", flush=True)
        answer += token
    print("\n")
    return answer

ask("What did Ramesses II build?")


## Cell 6 — Batch Test

In [ ]:
test_queries = [
    ("Who was Khufu?",                      "IDENTITY",   "Khufu"),
    ("What did Ramesses II build?",          "MONUMENTS",  "Ramesses II"),
    ("How did Cleopatra die?",               "DEATH",      "Cleopatra VII"),
    ("Tell me about Djoser",                 "IDENTITY",   "Djoser"),
    ("Who were Hatshepsut's children?",      "FAMILY",     "Hatshepsut"),
    ("What was Khufu famous for?",           "LEGACY",     "Khufu"),
    ("What battles did Ramesses II fight?",  "REIGN",      "Ramesses II"),
    ("What did Thutmose III conquer?",       "REIGN",      "Thutmose III"),
    ("How did Tutankhamun die?",             "DEATH",      "Tutankhamun"),
    ("What did Hatshepsut build?",           "MONUMENTS",  "Hatshepsut"),
]

print(f"{'Query':45} {'Exp':10} {'Got Pharaoh':20} {'Got Topic':12} Match")
print("─" * 100)
correct = 0
for query, exp_topic, exp_pharaoh in test_queries:
    results = retrieve(query, k=1)
    if results:
        r          = results[0]
        got_pharaoh = r["pharaoh"]
        got_topic   = r["topic"]
        pharaoh_ok  = exp_pharaoh.lower() in got_pharaoh.lower()
        topic_ok    = got_topic == exp_topic
        icon        = "✅" if pharaoh_ok and topic_ok else ("⚠️ " if pharaoh_ok else "❌")
        if pharaoh_ok and topic_ok: correct += 1
        print(f"  {query[:43]:45} {exp_topic:10} {got_pharaoh:20} {got_topic:12} {icon}")
    else:
        print(f"  {query[:43]:45} {exp_topic:10} {'NO RESULT':20} {'—':12} ❌")

print(f"\nScore: {correct}/{len(test_queries)}")


## Cell 7 — Multi-turn Chat

In [ ]:
history = []

def chat(query):
    global history
    docs    = retrieve(query, k=2)
    context = "\n\n---\n\n".join(d["text"] for d in docs)
    messages = [{"role": "system", "content": f"{SYSTEM_PROMPT}\n\nCONTEXT:\n{context}"}]
    for turn in history[-4:]:
        messages.append(turn)
    messages.append({"role": "user", "content": query})

    print(f"You: {query}")
    print(f"Bot: ", end="")
    answer = ""
    for chunk in ollama.chat(model=LLM_MODEL, messages=messages, stream=True,
                              options={"num_predict": 400, "temperature": 0.2}):
        token = chunk["message"]["content"]
        print(token, end="", flush=True)
        answer += token
    print("\n")
    history.append({"role": "user",      "content": query})
    history.append({"role": "assistant", "content": answer})

history = []
chat("Who was Cleopatra?")
chat("How did she die?")
chat("Tell me about Ramesses II")


## Cell 8 — Inspect Chunk

In [ ]:
def inspect(query, k=3):
    results = retrieve(query, k=k, verbose=True)
    print(f"\nQuery: {query}\n{'='*60}")
    for i, r in enumerate(results):
        print(f"\nChunk {i+1}")
        print(f"  Pharaoh : {r['pharaoh']}")
        print(f"  Topic   : {r['topic']}")
        print(f"  Anchor  : {r['is_anchor']}")
        print(f"\n{r['text']}\n")

inspect("What did Ramesses II build?")
